In [ ]:
# Fast Parallel + Vectorised Version with Real-Time Progress Bar
# GT-explicit, robust version

import pandas as pd
import itertools
import os
import logging
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm
os.chdir("/tmp")
# ---------------------------#
#       CONFIGURATION        #
# ---------------------------#

input_folder = "Individual_data_Uniti_generated_features_withDC_with_Adjusted AF for CRI_cohort_new"
output_folder = os.path.join(input_folder, "inds_rareAFCounts_Uniti_cohort_new")
os.makedirs(output_folder, exist_ok=True)

# ---------------------------#
#       LOGGING SETUP        #
# ---------------------------#

log_file = os.path.join(output_folder, "processing_log.txt")
logging.basicConfig(
    filename=log_file,
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

console = logging.StreamHandler()
console.setLevel(logging.INFO)
formatter = logging.Formatter("%(levelname)s: %(message)s")
console.setFormatter(formatter)
logging.getLogger().addHandler(console)

logging.info("===== Starting TSV modification with Rare_AF_Pair_Count =====")
logging.info(f"Input folder: {input_folder}")
logging.info(f"Output folder: {output_folder}")

# ---------------------------#
#    PROCESSING FUNCTION     #
# ---------------------------#

def process_file(file_path):
    start = time.time()
    filename = os.path.basename(file_path)
    base_name = os.path.splitext(filename)[0]
    logging.info(f"Started processing: {filename}")

    try:
        df = pd.read_csv(file_path, sep="\t", low_memory=False)
    except Exception as e:
        return f"❌ Failed to read {filename}: {e}"

    # ---------------------------
    # REQUIRED COLUMNS CHECK
    # ---------------------------
    required_cols = {"GT", "AF", "SYMBOL"}
    missing = required_cols - set(df.columns)
    if missing:
        return f"⚠️ Skipping {filename}: Missing columns {missing}"

    genotype_col = "GT"

    # ---------------------------
    # FILTER: HET + RARE AF
    # ---------------------------

    gt_series = df[genotype_col].astype(str)

    # Heterozygous: allow both 0/1 and 1/0
    het_mask = gt_series.isin(["0/1", "1/0"])

    # Rare AF threshold
    rare_mask = df["AF"] < 0.01

    df_filtered = df[het_mask & rare_mask]

    # ---------------------------
    # CANDIDATE GENES (≥2 variants)
    # ---------------------------

    gene_variant_counts = df_filtered["SYMBOL"].value_counts()
    candidate_genes = gene_variant_counts[gene_variant_counts >= 2].index.tolist()

    if not candidate_genes:
        # Still save file with Rare_AF_Pair_Count = 0
        df["Rare_AF_Pair_Count"] = 0
        out_path = os.path.join(output_folder, f"{base_name}.tsv")
        df.to_csv(out_path, sep="\t", index=False)
        elapsed = time.time() - start
        return f"ℹ️ No candidate genes in {filename} ({elapsed:.2f} sec)"

    # ---------------------------
    # COMPUTE Rare_AF_Pair_Count
    # ---------------------------

    af_product_counts = {}

    for gene in candidate_genes:
        af_values = (
            df_filtered.loc[df_filtered["SYMBOL"] == gene, "AF"]
            .dropna()
            .astype(float)
            .tolist()
        )

        if len(af_values) < 2:
            continue

        qualifying_pairs = [
            1
            for af1, af2 in itertools.combinations(af_values, 2)
            if af1 * af2 <= 0.001
        ]

        af_product_counts[gene] = len(qualifying_pairs)

    # ---------------------------
    # ADD COLUMN + SAVE
    # ---------------------------

    df["Rare_AF_Pair_Count"] = (
        df["SYMBOL"].map(af_product_counts).fillna(0).astype(int)
    )

    output_path = os.path.join(output_folder, f"{base_name}.tsv")
    try:
        df.to_csv(output_path, sep="\t", index=False)
    except Exception as e:
        return f"❌ Failed to save {filename}: {e}"

    elapsed = time.time() - start
    return f"✅ Finished {filename} in {elapsed:.2f} sec → Saved"

# ---------------------------#
#    PARALLEL FILE LOOP      #
# ---------------------------#

if __name__ == "__main__":
    files = [
        os.path.join(input_folder, f)
        for f in os.listdir(input_folder)
        if f.endswith(".tsv")
    ]

    with ProcessPoolExecutor() as executor:
        futures = []
        with tqdm(
            total=len(files),
            desc="Processing files",
            dynamic_ncols=True,
            unit="file"
        ) as pbar:

            for f in files:
                futures.append(executor.submit(process_file, f))
                pbar.update(1)

            for future in as_completed(futures):
                try:
                    result = future.result()
                    logging.info(result)
                except Exception as e:
                    logging.error(f"Unhandled error: {e}")

    logging.info("🎉 All files processed and saved with Rare_AF_Pair_Count.")
